# Burmese-English Machine Translation (A3 Project)

**Student**: Htut Ko Ko  
**Course**: Natural Language Understanding  
**Task**: Burmese (my) <-> English (en) Translation using Transformer

## Project Overview
This notebook implements a Neural Machine Translation system using a **Transformer** architecture. 
We use the **ALT (Asian Language Treebank)** dataset for Burmese-English parallel data.
We use **SentencePiece** for subword tokenization to handle the Burmese script effectively.

## Pipeline
1.  **Setup**: Install/Import dependencies.
2.  **Data Loading**: Load the ALT dataset.
3.  **Tokenization**: Train SentencePiece model on the corpus.
4.  **Data Processing**: Create PyTorch Datasets and DataLoaders.
5.  **Model**: Implement Transformer (using `nn.Transformer`).
6.  **Training**: Train the model and log performance.
7.  **Evaluation**: Calculate BLEU score on Test set.
8.  **Inference**: Demo function and save model for Web App.

## 1. Setup and Imports

In [ ]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
# Install dependencies if missing (uncomment if needed)
# !pip install sentencepiece datasets portalocker

## 2. Data Loading (ALT Dataset)
We will use the **ALT (Asian Language Treebank)** dataset via the HuggingFace `datasets` library.

In [ ]:
from datasets import load_dataset

print("Loading ALT Dataset (Burmese-English)...")
try:
    # Load ALT dataset from HuggingFace (my-en pair)
    # Note: 'alt' dataset on HF might need specific config configuration or we can use 'bs-modeling-metadata/alt-burmese-english-parallel'
    # For reliability, we'll try to load a known good source or fallback to manual download if needed.
    # Here we use 'larryvrh/alt-my-en' or similar if available, else we process raw files if local.
    # Let's try loading 'alt' configuration directly if supported, otherwise 'Helsinki-NLP/alt' does not exist.
    # Using a generic approach: Loading from a known reliable HF path or url if standard 'alt' fails.
    
    # Let's use 'my_alt' from 'Asian-Language-Treebank' if available, but for now we'll assume the user has internet access.
    # We will use 'alt' script if available or a direct parquet/csv link if we were doing custom.
    # Actually, let's use the 'alt' dataset provided by 'my_en' config if possible.
    
    dataset = load_dataset("alt", split="train+validation+test") # Load all for custom splitting
    print(f"Loaded {len(dataset)} sentences from ALT dataset.")
    
    # Filter/Extract only Burmese and English
    data = []
    for item in dataset:
        # ALT structure usually: {'translation': {'bg': '...', 'en': '...', 'my': '...'}}
        # The HF 'alt' dataset structure check:
        if 'translation' in item:
            if 'my' in item['translation'] and 'en' in item['translation']:
                data.append({
                    'my': item['translation']['my'],
                    'en': item['translation']['en']
                })
    
    print(f"Extracted {len(data)} Burmese-English pairs.")
    
except Exception as e:
    print(f"Error loading from HF: {e}")
    print("Attempting fallback or assuming local file 'alt_my_en.csv' exists...")
    # fallback code would go here


In [ ]:
# Convert to DataFrame for easier handling
df = pd.DataFrame(data)
print(df.head())

# Basic Cleaning
# 1. Drop NaN/None
df = df.dropna(subset=['my', 'en'])
# 2. Ensure they are strings
df['my'] = df['my'].astype(str)
df['en'] = df['en'].astype(str)

# 3. Remove empty strings
df = df[df['my'].str.strip() != '']
df = df[df['en'].str.strip() != '']
print(f"After cleaning: {len(df)} pairs")

print("\n--- Data Alignment Check ---")
for i in range(5):
    sample = df.sample(1).iloc[0]
    print(f"Source (my): {sample['my']}")
    print(f"Target (en): {sample['en']}")
    print("-" * 20)

## 3. Tokenization (SentencePiece)
Burmese does not use spaces between words cleanly. **SentencePiece** is excellent for this as it builds a vocabulary based on subword frequency, handling rare words and no-space languages effectively without external segmenters.

In [ ]:
import sentencepiece as spm

# 1. Save texts to files to train tokenizer
with open('train_my.txt', 'w', encoding='utf-8') as f:
    for line in df['my']:
        f.write(line + '\n')

with open('train_en.txt', 'w', encoding='utf-8') as f:
    for line in df['en']:
        f.write(line + '\n')

# 2. Train SentencePiece models
vocab_size = 4000 # Reduced for small dataset (~20k sentences) to learn better representations
model_type = 'bpe' # Byte-Pair Encoding

print("Training Burmese Tokenizer...")
spm.SentencePieceTrainer.train(
    input='train_my.txt', 
    model_prefix='spm_my', 
    vocab_size=vocab_size, 
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Training English Tokenizer...")
spm.SentencePieceTrainer.train(
    input='train_en.txt', 
    model_prefix='spm_en', 
    vocab_size=vocab_size, 
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Tokenizer training complete!")

In [ ]:
# Load the processors
sp_my = spm.SentencePieceProcessor(model_file='spm_my.model')
sp_en = spm.SentencePieceProcessor(model_file='spm_en.model')

# Test Tokenization
idx = 0
print(f"Original my: {df.iloc[idx]['my']}")
print(f"Tokens: {sp_my.encode(df.iloc[idx]['my'], out_type=str)}")
print(f"IDs: {sp_my.encode(df.iloc[idx]['my'], out_type=int)}")

print(f"\nOriginal en: {df.iloc[idx]['en']}")
print(f"Tokens: {sp_en.encode(df.iloc[idx]['en'], out_type=str)}")
print(f"IDs: {sp_en.encode(df.iloc[idx]['en'], out_type=int)}")

## 4. PyTorch Dataset and DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_trg):
        self.data = df
        self.sp_src = sp_src
        self.sp_trg = sp_trg
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['my']
        trg_text = self.data.iloc[idx]['en']
        
        # Encode with EOS
        # spm doesn't add sos/eos by default unless configured, we'll adds manually for safety or usage in model
        # Use bos_id() for beginning of sentence
        src_ids = [self.sp_src.bos_id()] + self.sp_src.encode(src_text, out_type=int) + [self.sp_src.eos_id()]
        trg_ids = [self.sp_trg.bos_id()] + self.sp_trg.encode(trg_text, out_type=int) + [self.sp_trg.eos_id()]
        
        return torch.tensor(src_ids), torch.tensor(trg_ids)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
        
    # Pad sequences
    # PAD ID is 0 for our spm models
    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=0)
    trg_pad = pad_sequence(trg_batch, batch_first=True, padding_value=0)
    
    return src_pad, trg_pad

# Split Data
train_df = df.sample(frac=0.8, random_state=SEED)
val_test_df = df.drop(train_df.index)
val_df = val_test_df.sample(frac=0.5, random_state=SEED)
test_df = val_test_df.drop(val_df.index)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

train_dataset = TranslationDataset(train_df, sp_my, sp_en)
val_dataset = TranslationDataset(val_df, sp_my, sp_en)
test_dataset = TranslationDataset(test_df, sp_my, sp_en)

BATCH_SIZE = 64 # Increased to stabilize gradients
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 5. Transformer Model
Using PyTorch's `nn.Transformer`.

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, 
                 d_model=512, nhead=8, num_encoder_layers=3, 
                 num_decoder_layers=3, dim_feedforward=2048, dropout=0.1, pad_idx=0):
        super(TransformerModel, self).__init__()
        
        self.d_model = d_model
        self.pad_idx = pad_idx
        
        # Embeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # Transformer
        self.transformer = nn.Transformer(
            d_model=d_model, 
            nhead=nhead, 
            num_encoder_layers=num_encoder_layers, 
            num_decoder_layers=num_decoder_layers, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            batch_first=True
        )
        
        # Output Layer
        self.fc_out = nn.Linear(d_model, trg_vocab_size)
        
        self.init_weights()
    
    def init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        
    def forward(self, src, trg):
        # src: [batch_size, src_len]
        # trg: [batch_size, trg_len]
        
        # Create masks
        src_key_padding_mask = (src == self.pad_idx)
        trg_key_padding_mask = (trg == self.pad_idx)
        
        # Target mask for autoregressive decoding (prevent peeking future)
        trg_mask = self.transformer.generate_square_subsequent_mask(trg.size(1)).to(src.device)
        
        # Embed + Positional Encoding
        src_emb = self.src_embedding(src) * math.sqrt(self.d_model)
        trg_emb = self.trg_embedding(trg) * math.sqrt(self.d_model)
        
        src_emb = self.pos_encoder(src_emb)
        trg_emb = self.pos_encoder(trg_emb)
        
        # Transformer Forward
        output = self.transformer(
            src=src_emb, 
            tgt=trg_emb, 
            tgt_mask=trg_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=trg_key_padding_mask
        )
        
        prediction = self.fc_out(output)
        return prediction

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

## 6. Training Loop

In [ ]:
# Config
SRC_VOCAB_SIZE = vocab_size
TRG_VOCAB_SIZE = vocab_size
D_MODEL = 256
N_HEAD = 4  # Reduced for small dataset
NUM_LAYERS = 2 # Reduced layers
FF_DIM = 512
DROPOUT = 0.4  # Increased for regularization
LR = 0.0005
EPOCHS = 100 # Increased to allow convergence

model = TransformerModel(SRC_VOCAB_SIZE, TRG_VOCAB_SIZE, D_MODEL, N_HEAD, NUM_LAYERS, NUM_LAYERS, FF_DIM, DROPOUT, pad_idx=0).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1) # Label smoothing helps with generation

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        
        # trg input = trg[:, :-1] (all except last)
        # trg output = trg[:, 1:] (all except first - predicted next token)
        output = model(src, trg[:, :-1])
        
        output_dim = output.shape[-1]
        
        # Flatten for loss calculation
        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg[:, :-1])
            
            output_dim = output.shape[-1]
            output = output.contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)
            
            loss = criterion(output, trg)
            epoch_loss += loss.item()
            
    return epoch_loss / len(iterator)

print("Starting training...")
best_valid_loss = float('inf')

for epoch in range(EPOCHS):
    start_time = time.time()
    
    train_loss = train(model, train_loader, optimizer, criterion, 1.0)
    valid_loss = evaluate(model, val_loader, criterion)
    
    end_time = time.time()
    
    # Step the scheduler
    scheduler.step(valid_loss)
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'transformer_model.pt')
    
    print(f'Epoch: {epoch+1:02} | Time: {end_time-start_time:.0f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')
    print(f'\t LR: {optimizer.param_groups[0]["lr"]:.6f}')

## 7. Inference and Verification

In [ ]:
# Load Best Model
model.load_state_dict(torch.load('transformer_model.pt', map_location=device))

def translate_sentence(sentence, model, sp_src, sp_trg, max_len=50, device=device):
    model.eval()
    
    # Tokenize src
    tokens = [sp_src.bos_id()] + sp_src.encode(sentence, out_type=int) + [sp_src.eos_id()]
    print(f"Debug - Source tokens: {sp_src.encode(sentence, out_type=str)}")
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)
    
    # Start with SOS
    trg_indices = [sp_trg.bos_id()]
    
    for i in range(max_len):
        trg_tensor = torch.LongTensor(trg_indices).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(src_tensor, trg_tensor)
        
        # Get last predicted token
        pred_token = output.argmax(2)[:, -1].item()
        
        trg_indices.append(pred_token)
        
        if pred_token == sp_trg.eos_id():
            break
            
    # Decode
    translated_text = sp_trg.decode(trg_indices)
    return translated_text

# Test Translation
idx = random.randint(0, len(test_df)-1)
src_sent = test_df.iloc[idx]['my']
trg_sent = test_df.iloc[idx]['en']

print(f"Source: {src_sent}")
print(f"Target: {trg_sent}")
print(f"Pred: {translate_sentence(src_sent, model, sp_my, sp_en)}")

In [ ]:
# Save artifacts for Web App
# Already saved: 'transformer_model.pt', 'spm_my.model', 'spm_en.model'
# The web app will need these files.
import shutil

os.makedirs('app/models', exist_ok=True)
shutil.copy('transformer_model.pt', 'app/models/transformer_model.pt')
shutil.copy('spm_my.model', 'app/models/spm_my.model')
shutil.copy('spm_en.model', 'app/models/spm_en.model')
print("Models copied to app/models/")